# Sales Data Analysis (Python + SQL)

**Objective:** Clean a retail sales dataset, then use Pandas and SQL to find which regions, products and discount levels drive profit or loss.

**Tools:** Python, Pandas, SQL (SQLite), Matplotlib

**Dataset:** `superstore_sales_sample.csv` is a practice dataset in the style of the Superstore sales data (3,035 rows, 2023-2025). It intentionally contains duplicates, missing values and inconsistent text so that the cleaning steps are real.

## 1. Load the data

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt

df = pd.read_csv("superstore_sales_sample.csv")
print("Rows, Columns:", df.shape)
df.head()

## 2. Check data quality

In [ ]:
# Data quality checks
print("Missing values:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nRegion values (note the inconsistent spelling):")
print(df["region"].value_counts())

## 3. Clean the data

In [ ]:
# 1. Fix inconsistent text in region
df["region"] = df["region"].str.strip().str.title()

# 2. Remove duplicate rows
df = df.drop_duplicates()

# 3. Rows with no sales value cannot be analysed, so drop them
df = df.dropna(subset=["sales"])

# 4. Missing discount means no discount was given
df["discount"] = df["discount"].fillna(0)

# 5. Convert dates from text to datetime
df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"] = pd.to_datetime(df["ship_date"])

# 6. New columns for analysis
df["year"] = df["order_date"].dt.year
df["month"] = df["order_date"].dt.to_period("M").astype(str)
df["shipping_days"] = (df["ship_date"] - df["order_date"]).dt.days
df["profit_margin_pct"] = (df["profit"] / df["sales"] * 100).round(2)

print("Clean shape:", df.shape)
print(df["region"].unique())

## 4. Key numbers (KPIs)

In [ ]:
# Key numbers (KPIs)
total_sales = df["sales"].sum()
total_profit = df["profit"].sum()
print(f"Total Sales : {total_sales:,.0f}")
print(f"Total Profit: {total_profit:,.0f}")
print(f"Profit Margin: {total_profit/total_sales*100:.1f}%")
print(f"Orders: {df['order_id'].nunique()}")
print(f"Avg shipping days: {df['shipping_days'].mean():.1f}")

## 5. Analysis with Pandas

In [ ]:
# Analysis with Pandas
by_region = df.groupby("region")[["sales", "profit"]].sum().round(0).sort_values("profit", ascending=False)
print(by_region)
print()
by_cat = df.groupby("category")[["sales", "profit"]].sum().round(0)
print(by_cat)

## 6. Analysis with SQL

In [ ]:
# Load the clean data into an in-memory SQL database
conn = sqlite3.connect(":memory:")
df.assign(order_date=df["order_date"].astype(str), ship_date=df["ship_date"].astype(str)).to_sql("sales", conn, index=False)

def run(query):
    return pd.read_sql_query(query, conn)

In [ ]:
# Q1: Which sub-categories make the most and least profit?
run("""
SELECT sub_category,
       ROUND(SUM(sales), 0)  AS total_sales,
       ROUND(SUM(profit), 0) AS total_profit
FROM sales
GROUP BY sub_category
ORDER BY total_profit DESC
""")

In [ ]:
# Q2: How does discount level affect profit?
run("""
SELECT discount,
       COUNT(*) AS orders,
       ROUND(SUM(profit), 0) AS total_profit,
       ROUND(SUM(profit) * 100.0 / SUM(sales), 1) AS margin_pct
FROM sales
GROUP BY discount
ORDER BY discount
""")

In [ ]:
# Q3: Top 5 customers by sales
run("""
SELECT customer_id, ROUND(SUM(sales), 0) AS total_sales, COUNT(*) AS orders
FROM sales
GROUP BY customer_id
ORDER BY total_sales DESC
LIMIT 5
""")

In [ ]:
# Q4: Which months have the highest sales?
run("""
SELECT strftime('%m', order_date) AS month_number,
       ROUND(SUM(sales), 0) AS total_sales
FROM sales
GROUP BY month_number
ORDER BY total_sales DESC
LIMIT 5
""")

## 7. Visualizations

In [ ]:
# Chart 1: Monthly sales trend
monthly = df.groupby("month")["sales"].sum()
plt.figure(figsize=(10, 4))
plt.plot(monthly.index, monthly.values, marker="o")
plt.xticks(range(0, len(monthly), 3), monthly.index[::3], rotation=45)
plt.title("Monthly Sales Trend")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: Profit by sub-category
sub = df.groupby("sub_category")["profit"].sum().sort_values()
colors = ["red" if v < 0 else "green" for v in sub.values]
plt.figure(figsize=(8, 5))
plt.barh(sub.index, sub.values, color=colors)
plt.title("Profit by Sub-Category")
plt.xlabel("Profit")
plt.tight_layout()
plt.show()

In [ ]:
# Chart 3: Discount vs average profit margin
disc = df.groupby("discount")["profit_margin_pct"].mean()
plt.figure(figsize=(7, 4))
plt.bar([f"{int(d*100)}%" for d in disc.index], disc.values)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Average Profit Margin by Discount")
plt.xlabel("Discount")
plt.ylabel("Profit margin %")
plt.tight_layout()
plt.show()

## 8. Key insights

1. **Overall profit margin is only about 4.6%** on roughly 1.99M in sales, so profit is thin compared to sales.
2. **Discounts are the biggest problem.** Orders with no discount earn a 17.4% margin, while orders with 30% or more discount lose money (-10.5% at 30%, -29.2% at 50%). Discounts above 20% should be limited.
3. **Furniture loses money overall** (about -38K profit). Tables and Bookcases lose around 18K each despite high sales.
4. **Machines are the worst sub-category** (about -26K) even with 354K in sales, while **Copiers are the most profitable** (about 146K).
5. **West is the top region** by sales and profit. South has the lowest sales and profit.
6. **Sales peak in November and December**, so stock and campaigns should be planned for these months.

## 9. Recommendations
- Cap discounts at 20% for Furniture and Machines.
- Review pricing and costs for Tables, Bookcases and Machines.
- Focus marketing on Copiers and high-margin Technology items.
- Prepare inventory and staffing ahead of Nov-Dec.